<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Costos de transporte interno y externo de los productos priorizados

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.

In [1]:
import pandas as pd
import numpy as np

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import temp, tables

### Parámetros

Define los grupos de destino que entran a la comparación de competidores, el número de
competidores que se conserva por mercado y el corte del tercil de menor costo.

In [3]:
grupos_destino = ['Unión Europea','APEC Asia','América Latina y el Caribe','Estados Unidos','Reino Unido']

top_competidores = 30

corte_tercil = 1/3

### Lectura de datos

Carga la cartera con tamaño de mercado, las declaraciones agregadas del formulario 600, los
costos internos del registro de carga, el comercio bilateral y los costos externos de
transporte.

In [4]:
df_products = pd.read_parquet(temp / 'df_products_mercado.parquet')
df_products_externos = df_products.copy()

df_600 = pd.read_parquet(temp / 'df_600.parquet')
df_rndc = pd.read_parquet(temp / 'df_rndc.parquet')
df_comercio = pd.read_parquet(temp / 'df_comercio.parquet')
df_costos_externos = pd.read_parquet(temp / 'df_costos_externos.parquet')

carpeta_localizacion = tables / 'localizacion'
carpeta_localizacion.mkdir(parents=True, exist_ok=True)

### Costo interno por kilogramo

Promedia el costo del transporte terrestre por código de mercancía y lo asigna a cada producto
por su partida de cuatro dígitos. Los productos sin dato toman el promedio de su capítulo de
dos dígitos.

In [5]:
df_rndc = df_rndc.groupby(['CODMERCANCIA'])[['COSTO POR KG']].mean().reset_index().sort_values(['COSTO POR KG'], ascending=False)
df_rndc.columns = ['CODMERCANCIA', 'COSTO POR KG']

df_products['HS4'] = df_products['HS6'].astype(str).str[:4]
df_products['HS2'] = df_products['HS6'].astype(str).str[:2]
df_products = df_products.merge(df_rndc, left_on=['HS4'], right_on=['CODMERCANCIA'], how='left').drop(columns=['CODMERCANCIA']).sort_values(by='COSTO POR KG', ascending=False)
df_products = df_products.merge(
    df_products[['HS2','COSTO POR KG']].drop_duplicates().sort_values(by='HS2', ascending=False).groupby(['HS2'])[['COSTO POR KG']].mean().reset_index().round(2),
    on='HS2',
    how='left'
)
df_products['COSTO POR KG'] = df_products['COSTO POR KG_x'].fillna(df_products['COSTO POR KG_y'])
df_products = df_products.drop(columns=['COSTO POR KG_x', 'COSTO POR KG_y'])

df_products.round(2).to_excel(carpeta_localizacion / 'loc internos.xlsx', index=False)

### Competidores relevantes por mercado de destino

Identifica, para cada partida de cuatro dígitos y mercado de destino, los treinta mayores
proveedores con origen fuera del propio grupo, y añade a Colombia como competidor.

In [6]:
df_comercio = df_comercio.merge(
    df_comercio[['Importador','Grupo Importador']].drop_duplicates().rename(columns={'Importador': 'Exportador','Grupo Importador': 'Grupo Exportador'}),
    how='left'
)
df_paises_competidores = df_comercio.query("`Grupo Importador` in @grupos_destino and `Grupo Importador` != `Grupo Exportador`").groupby(['HS4','Grupo Importador','ISO Exportador','ISO Importador'])['Valor Comercio'].sum().reset_index().sort_values(by=['HS4','Grupo Importador','ISO Importador','Valor Comercio'], ascending=False)
df_paises_competidores['Ranking'] = df_paises_competidores.groupby(['HS4', 'Grupo Importador','ISO Importador']).cumcount() + 1
df_paises_competidores = df_paises_competidores.query("Ranking <= @top_competidores").drop(columns=['Valor Comercio','Ranking'])

df_paises_competidores_col = df_paises_competidores[['HS4','Grupo Importador','ISO Importador']].drop_duplicates().reset_index(drop=True)
df_paises_competidores_col['ISO Exportador'] = 'COL'

df_competidores = pd.concat([df_paises_competidores, df_paises_competidores_col], ignore_index=True).drop_duplicates()

### Ventaja en costos externos de transporte

Ordena a los competidores por costo por kilogramo dentro de cada partida, modo de transporte y
mercado, y cuenta en cuántos grupos de destino Colombia queda en el tercil más barato.

In [7]:
df_costos_externos_agg = df_costos_externos.query("`Grupo Importador` != 'Otro'").sort_values(by=['HS4','Grupo Importador'], ascending=False).reset_index(drop=True)
df_costos_externos_agg = df_costos_externos_agg.merge(df_competidores, on=['HS4','Grupo Importador','ISO Importador','ISO Exportador'], how='inner').round(3).sort_values(by=['HS4','Transporte','Grupo Importador','ISO Importador','Valor por kg'], ascending=False)
df_costos_externos_agg = df_costos_externos_agg.sort_values(by=['HS4','Transporte','Grupo Importador','ISO Importador','Valor por kg'], ascending=True)
df_costos_externos_agg['Rank'] = (df_costos_externos_agg
    .sort_values(by=['HS4','Transporte','Grupo Importador','ISO Importador','Valor por kg'], ascending=True)
    .groupby(['HS4','Transporte','ISO Importador']).cumcount() + 1
    )
df_costos_externos_agg['Max Rank'] = (df_costos_externos_agg
    .sort_values(by=['HS4','Transporte','Grupo Importador','ISO Importador','Valor por kg'], ascending=True)
    .groupby(['HS4','Transporte','ISO Importador'])['Rank'].transform('max')
    )
df_costos_externos_agg['Tercil'] = df_costos_externos_agg['Rank'] / df_costos_externos_agg['Max Rank']
df_costos_externos_agg = df_costos_externos_agg.query("`ISO Exportador` == 'COL' and Tercil <= @corte_tercil")
df_costos_externos_agg = df_costos_externos_agg.groupby(['HS4','Transporte'])['Grupo Importador'].nunique().reset_index().sort_values(by=['HS4','Transporte'], ascending=True)
df_costos_externos_agg = df_costos_externos_agg.pivot(index='HS4', columns='Transporte', values='Grupo Importador').reset_index().fillna(0)

df_products_externos['HS4'] = df_products_externos['HS6'].astype(str).str[:4]
df_products_externos = df_products_externos.merge(df_costos_externos_agg, on='HS4', how='left').fillna(0)

df_products_externos.to_excel(carpeta_localizacion / 'loc externos.xlsx', index=False)